# quantum-cortex — the recurrent base hybrid: 4 arms x 8 seeds x 2 tiers (RESUMABLE)

Rev38: L3 as the base, the attention as the added path (zero init). Four arms declared before the run: **bare** (no protection: the direct measure of pre emption), **critical** (the gate opens on a measured plateau), **cls** (complementary systems: consolidation of the trunk on the full model, geometric retraction of the gate, ratios 1/2), **cls-phi** (the same with φ where a ratio is needed). 64 units at 1500 steps; the references (L3 pure, the control) come from the committed `metrics/mqar/ladder8-ckpt/` units, same seeds, tiers, steps: every test is paired on seed. Estimated 8 to 9 h on T4; fits one session with `--time-budget-min 700`; if cut, resume.

Run this AFTER the Σ run and after REPRISE step 4 (order A). GPU T4, Internet On. **Download the log as a file before reading any number.**


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema

In [ ]:
# Find previous checkpoint dirs mounted as Inputs (resume). Empty list = fresh start.
import glob, os
PREV = sorted(set(glob.glob('/kaggle/input/**/rb-ckpt', recursive=True)))
print('resume_from:', PREV if PREV else '(none — fresh start)')
if PREV:
    n = sum(len(glob.glob(p + '/unit-*.json')) for p in PREV)
    print(f'{n} finished units found in previous versions — they will be reused, not recomputed')


In [ ]:
resume = ' '.join(f'--resume-from {p}' for p in PREV)
!python -m cortex_eval.resumable_rb --steps 1500 --seeds 1337 2024 7 42 99 3 11 2026 --reference-from metrics/mqar/ladder8-ckpt --time-budget-min 700 {resume}


In [ ]:
import json, os
d = 'metrics/mqar/rb-ckpt'
if os.path.exists(d + '/LATEST-rb.json'):
    r = json.load(open(d + '/LATEST-rb.json'))
    print('COMPLETE —', r['units_done'], '/', r['units_total'])
    for k, v in r['gate_readout'].items(): print(f'  {k:36s} {v}')
    for k, st in r['stats'].items(): print(k, json.dumps(st))
    print('Commit metrics/mqar/rb-ckpt/ (all unit-*.json + LATEST) — the units ARE the record.')
else:
    r = json.load(open(d + '/PROGRESS-rb.json')); print(f"PARTIAL — {r['units_done']} / {r['units_total']} units. New version with this Output as Input → Run All to resume.")
